# M2 — Baseline Reproduction (InceptionV3)

End-to-end notebook: load the Kaggle/Msoud **Brain Tumor MRI** dataset → train ImageNet-pretrained **InceptionV3** → evaluate with accuracy, precision, recall, F1 (macro + per-class), AUC, and confusion matrix.

**Framework:** TensorFlow / Keras (`keras.applications.InceptionV3`).

**Split (fixed before training):** official `Training/` / `Testing/` folders; 20% of `Training/` held out as validation (seed=42). `Testing/` is the final test set.

**No clinical claims** — research metrics only; always read recall + AUC with accuracy.


In [ ]:
# If running on Colab/Kaggle, uncomment and install:
# !pip install -q tensorflow scikit-learn matplotlib seaborn pillow pandas
import os, sys
from pathlib import Path

# Repo root = parent of notebooks/
ROOT = Path.cwd()
if (ROOT / 'src').is_dir():
    pass
elif (ROOT.parent / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.mplconfig'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import tensorflow as tf
print('TF', tf.__version__)
print('Devices', tf.config.list_physical_devices())


## 0. Data location

Place the dataset so that `data/Training/{glioma,meningioma,notumor,pituitary}` and `data/Testing/...` exist.

- Kaggle: `kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset`
- Mirror used in this project: Zenodo `10.5281/zenodo.12735702` (same files, ~149 MB zip / ~155.8 MB listed)


In [ ]:
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / 'Training').is_dir() and (DATA_DIR / 'Testing').is_dir(), DATA_DIR


In [ ]:
from src.data_loader import CLASS_NAMES, get_train_val_test
from src.train_baseline import build_model, plot_history, unfreeze_top_blocks
from src.evaluate import evaluate_model
import json

BATCH_SIZE = 16
SEED = 42
HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 5

train_ds, val_ds, test_ds, meta = get_train_val_test(
    DATA_DIR, batch_size=BATCH_SIZE, val_split=0.2, seed=SEED
)
print(json.dumps(meta, indent=2))


## 1. Build model

InceptionV3 ImageNet backbone (`include_top=False`) + GAP + Dense(256) + Dropout + 4-class softmax.

**Phase 1:** freeze backbone, train head (Adam 1e-3).  
**Phase 2:** unfreeze last 50 backbone layers, fine-tune (Adam 1e-5).


In [ ]:
tf.keras.utils.set_random_seed(SEED)
model = build_model(num_classes=len(CLASS_NAMES))
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


In [ ]:
callbacks_head = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
]
print('=== Phase 1: frozen backbone ===')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=callbacks_head)


In [ ]:
print('=== Phase 2: fine-tune top blocks ===')
unfreeze_top_blocks(model, n_unfreeze=50)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
ckpt_dir = RESULTS_DIR / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_dir / 'best_inceptionv3.keras'),
            monitor='val_accuracy',
            save_best_only=True,
        ),
    ],
)
plot_history([h1.history, h2.history], RESULTS_DIR / 'M2_training_curves.png')
model.save(ckpt_dir / 'final_inceptionv3.keras')
print('Saved curves + checkpoint')


## 2. Evaluate on official Testing/ split

Reports accuracy, macro precision/recall/F1, AUC, per-class breakdown, confusion matrix.


In [ ]:
protocol = {
    **meta,
    'phase1': 'backbone frozen, Adam 1e-3',
    'phase2': 'unfreeze last 50 backbone layers, Adam 1e-5',
    'head_epochs_ran': len(h1.history['loss']),
    'finetune_epochs_ran': len(h2.history['loss']),
    'paper_target': {'accuracy': 0.9712, 'auc': 0.9984, 'recall': 0.9659},
}
metrics = evaluate_model(
    model, test_ds, class_names=list(CLASS_NAMES),
    results_dir=RESULTS_DIR, protocol=protocol,
)
metrics['accuracy'], metrics['recall_macro'], metrics['auc_macro_ovr']


### Limitations (required)

This is a research reproduction on a public MRI set. Scores are **not** clinical performance. The paper averages **5-fold CV** after heavier augmentation; this notebook uses the fixed Kaggle Testing folder — gaps vs 97.12% / 0.9984 / 0.9659 are expected and should be reported honestly.
